In [1]:

import os
import sys
import subprocess

# Install compatible packages once
if not os.path.exists("/content/day5_setup_done"):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain==1.5.2",
            "langchain-core==1.8.4",
            "langchain-groq==1.2.1",
            "langgraph",
            "chromadb"
        ],
        check=True
    )

    open("/content/day5_setup_done", "w").close()

    from google.colab import runtime
    runtime.restart_runtime()

# Imports
import langchain
import langchain_core
import langchain_groq
import langgraph
import chromadb
import csv

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("Imports successful!")
print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

# API Key
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Please add GROQ_API_KEY in Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Grocery catalogue
PRODUCTS = {
    "rice": 60,
    "sugar": 45,
    "milk": 30,
    "bread": 40,
    "eggs": 7,
    "apples": 120,
    "tomatoes": 35,
    "potatoes": 30,
    "oil": 150
}

# Price tool
@tool
def get_price(product: str, quantity: float = 1):
    """Get the price of a grocery product for a given quantity."""
    product = product.lower().strip()

    if product not in PRODUCTS:
        return "Product not found. Available products: " + ", ".join(PRODUCTS)

    if quantity <= 0:
        return "Quantity must be greater than zero."

    total = PRODUCTS[product] * quantity

    return f"{quantity} units of {product} cost Rs. {total:.2f}."

# Store hours tool
@tool
def is_open():
    """Check whether the grocery store is currently open."""
    return "Store timings have not been configured yet. Please contact the store for timings."

# Order tool
@tool
def log_order(customer_name: str, items: str, confirmed: bool):
    """Save a grocery order only when the customer confirms it."""
    if not confirmed:
        return "Order was not saved because it was not confirmed."

    file_name = "grocery_orders.csv"
    file_exists = os.path.exists(file_name)

    with open(file_name, "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(["Customer Name", "Items"])

        writer.writerow([customer_name, items])

    return "Order successfully saved."

# Business knowledge
PRODUCT_FACTS = [
    "Rice costs Rs. 60 per kg.",
    "Sugar costs Rs. 45 per kg.",
    "Milk costs Rs. 30 per litre.",
    "Bread costs Rs. 40 per packet.",
    "Eggs cost Rs. 7 per piece.",
    "Apples cost Rs. 120 per kg.",
    "Tomatoes cost Rs. 35 per kg.",
    "Potatoes cost Rs. 30 per kg.",
    "Oil costs Rs. 150 per litre.",
    "Customers can ask about grocery products, prices and shopping orders.",
    "Orders should only be saved after the customer confirms them."
]

# Add optional business facts file
if os.path.exists("business_facts.txt"):
    with open("business_facts.txt", "r", encoding="utf-8") as file:
        extra_facts = [
            line.strip() for line in file.readlines()
            if line.strip()
        ]
    PRODUCT_FACTS.extend(extra_facts)

# Chroma knowledge base
client = chromadb.Client()

try:
    client.delete_collection("grocery_knowledge")
except Exception:
    pass

collection = client.create_collection("grocery_knowledge")

collection.add(
    documents=PRODUCT_FACTS,
    ids=[f"fact_{i}" for i in range(len(PRODUCT_FACTS))]
)

# Knowledge search
def search_grocery(query):
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "No relevant information found."

    return "\n".join(documents)

# FAQ / RAG tool
@tool
def search_faq(question: str):
    """Search the grocery store knowledge base for relevant information."""
    return search_grocery(question)

# Agent instructions
SYSTEM = """
You are a helpful Smart Grocery Shopping Assistant.

Your responsibilities:
- Help customers with grocery shopping.
- Answer questions about products and prices.
- Use get_price to calculate product costs.
- Use search_faq to retrieve information from the grocery knowledge base.
- Use is_open when customers ask about store timings or whether the store is open.
- Help customers prepare orders.
- Before saving an order, collect the customer's name and item details.
- Save orders only when the customer clearly confirms.
- Use log_order to save confirmed orders.
- Never invent product prices or store policies.
- If information is unavailable, clearly say so.
- Respond in simple, friendly language.
"""

# Memory
memory = InMemorySaver()

# Create agent
agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order, search_faq],
    system_prompt=SYSTEM,
    checkpointer=memory
)

# Chat function
def chat(message, thread_id="grocery_customer_1"):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config=config
    )

    return result["messages"][-1].content

print("\nSmart Grocery Assistant is ready!")

# Testing
print("\n--- PRICE TEST ---")
print(chat("What is the price of 2 kg rice?"))

print("\n--- KNOWLEDGE BASE TEST ---")
print(chat("What products are available in the grocery store?"))

print("\n--- MEMORY TEST ---")
print(chat("My name is Anusha."))
print(chat("What is my name?"))

# Interactive chatbot
print("\nStart chatting with your Grocery Assistant!")
print("Type 'exit' to stop.")
print("Type 'new' to start a new conversation.")

thread_id = "grocery_customer_1"

while True:
    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    if user_input.lower() == "new":
        thread_id = "customer_" + str(len(os.listdir(".")))
        print("New conversation started.")
        continue

    if not user_input:
        continue

    try:
        response = chat(user_input, thread_id)
        print("\nAssistant:", response)

    except Exception as e:
        print("\nError:", str(e))

# Display saved orders
if os.path.exists("grocery_orders.csv"):
    print("\nSaved orders:")
    with open("grocery_orders.csv", "r", encoding="utf-8") as file:
        print(file.read())


Imports successful!
LangChain: 1.5.2
LangChain Core: 1.8.4

Smart Grocery Assistant is ready!

--- PRICE TEST ---
Sure! 2 kg of rice costs **Rs. 120.00**. Let me know if you’d like to add it to your cart or need anything else!

--- KNOWLEDGE BASE TEST ---
Here’s a quick snapshot of what we have in stock:

| Product | Price (per unit) |
|---------|------------------|
| Apples | Rs. 120 per kg |
| Bread | Rs. 40 per packet |

If you’re looking for something else—like rice, milk, or any other item—just let me know and I’ll check the price for you!

--- MEMORY TEST ---
Nice to meet you, Anusha! How can I help you with your grocery shopping today?
Your name is Anusha.

Start chatting with your Grocery Assistant!
Type 'exit' to stop.
Type 'new' to start a new conversation.

You: What is the price of 2 kg rice?

Assistant: 2 kg of rice costs **Rs. 120.00**. Let me know if you’d like to add it to your order or need anything else!

You: How much will 3 litres of oil cost?

Assistant: 3 litres o

In [ ]:
import sys
import subprocess

packages = [
    "langchain==1.5.2",
    "langchain-core==1.8.4",
    "langchain-groq==1.2.1",
    "langgraph",
    "chromadb"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True
)

print("Installation completed. Now restart the Colab session.")